In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm

In [ ]:
with h5py.File('nexus_outputs.h5', 'r') as f:
    norm_dens = f['z_0.0']['density'][:]
    voids = f['z_0.0']['voids'][:]
    filaments = f['z_0.0']['filaments'][:]
    nodes = f['z_0.0']['nodes'][:]
    walls = f['z_0.0']['walls'][:]
    print(f"Suessfully Loaded Stuff")

norm_dens = np.transpose(norm_dens, (1, 2, 0))
voids = np.transpose(voids, (1, 2, 0))
filaments = np.transpose(filaments, (1, 2, 0))
nodes = np.transpose(nodes, (1, 2, 0))
walls = np.transpose(walls, (1, 2, 0))

In [ ]:
basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

coords_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Coordinates'])/1000 #in cMpc/h

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (dl)

In [ ]:
slice_index = 0

z_min = slice_index * dl
z_max = (slice_index + 1) * dl
mask = (coords_0[:,2] > z_min) & (coords_0[:,2] < z_max)

x = coords_0[:, 0][mask]
y = coords_0[:, 1][mask]
z = coords_0[:, 2][mask]

fig = plt.figure(figsize=(12,4))

ax1 = fig.add_subplot(1,2,1)
im1 = ax1.imshow(np.log10(norm_dens[:,:,slice_index]),origin = "lower",cmap="inferno", extent = [0,35,0,35])
ax1.set_xlabel("cMpc/h")
ax1.set_ylabel("cMpc/h")
ax1.set_aspect('equal')
fig.colorbar(im1, ax = ax1, label = r"$log_{10}(1+\delta)$")

ax2 = fig.add_subplot(1,2,2)
counts, xedges, yedges, im2 = ax2.hist2d(x, y, bins = 500, cmap = "inferno", norm = LogNorm())
ax2.set_xlabel("cMpc/h")
ax2.set_ylabel("cMpc/h")
ax2.set_aspect('equal')
fig.colorbar(im2, ax = ax2, label = "Counts")

plt.tight_layout();

In [ ]:
ids_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['ParticleIDs'])
vels_0 = il.snapshot.loadSubset(basePath, 99, 'dm', fields = ['Velocities'])
print (vels_0) #km sqrt(a) /s

In [ ]:
bool_filament = filaments.astype(bool)
bool_wall = walls.astype(bool)
bool_node = nodes.astype(bool)
bool_void = voids.astype(bool)

grid_x = np.floor(coords_0[:, 0] / dl).astype(int) % res
grid_y = np.floor(coords_0[:, 1] / dl).astype(int) % res
grid_z = np.floor(coords_0[:, 2] / dl).astype(int) % res

in_node_mask = bool_node[grid_x, grid_y, grid_z]

node_particle_ids = ids_0[in_node_mask]
print(f"Total DM particles: {len(ids_0)}")
print(f"Particles in nodes: {len(node_particle_ids)}")

in_filament_mask = bool_filament[grid_x, grid_y, grid_z]

filament_particle_ids = ids_0[in_filament_mask]
print(f"Particles in filaments: {len(filament_particle_ids)}")

in_wall_mask = bool_wall[grid_x, grid_y, grid_z]

wall_particle_ids = ids_0[in_wall_mask]
print(f"Particles in walls: {len(wall_particle_ids)}")

in_void_mask = bool_void[grid_x, grid_y, grid_z]

void_particle_ids = ids_0[in_void_mask]
print(f"Particles in voids: {len(void_particle_ids)}")

In [ ]:
match_mask = np.isin(ids_0, node_particle_ids)
node_vels_0 = vels_0[match_mask]
print("Recovered valocities for particles in nodes")

match_mask = np.isin(ids_0, filament_particle_ids)
filament_vels_0 = vels_0[match_mask]
print("Recovered valocities for particles in filaments")

match_mask = np.isin(ids_0, wall_particle_ids)
wall_vels_0 = vels_0[match_mask]
print("Recovered valocities for particles in walls")

match_mask = np.isin(ids_0, void_particle_ids)
void_vels_0 = vels_0[match_mask]
print("Recovered valocities for particles in voids")

In [ ]:
node_speeds_0 = np.sqrt(node_vels_0[:,0]**2 + node_vels_0[:,1]**2 + node_vels_0[:,2]**2)
filament_speeds_0 = np.sqrt(filament_vels_0[:,0]**2 + filament_vels_0[:,1]**2 + filament_vels_0[:,2]**2)
wall_speeds_0 = np.sqrt(wall_vels_0[:,0]**2 + wall_vels_0[:,1]**2 + wall_vels_0[:,2]**2)
void_speeds_0 = np.sqrt(void_vels_0[:,0]**2 + void_vels_0[:,1]**2 + void_vels_0[:,2]**2)

In [ ]:
plt.hist(node_speeds_0, bins="scott", color = "orange", density=True, histtype = "step",label = "Nodes")
plt.hist(filament_speeds_0, bins="scott", color = "g", density=True, histtype = "step",label = "Filaments")
plt.hist(wall_speeds_0, bins="scott", color = "r", density=True, histtype = "step",label = "Walls")
plt.hist(void_speeds_0, bins="scott", color = "b", density=True, histtype = "step",label = "Voids")
plt.xlabel(r"km $\sqrt{a}$ / s")
plt.ylabel("PDF")
#plt.xscale("log")
plt.grid()
plt.title("Speed distribution at z=0")
plt.legend();

In [ ]:
snap_z = 0 

particles_z = il.snapshot.loadSubset(basePath, snap_z, 'dm', ['Coordinates', 'ParticleIDs', 'Velocities'])
coords_z = particles_z['Coordinates']/1000
ids_z = particles_z['ParticleIDs']
vels_z = particles_z['Velocities']

match_mask = np.isin(ids_z, node_particle_ids)
node_vels_z = vels_z[match_mask]
print("Recovered valocities for particles in nodes")

match_mask = np.isin(ids_z, filament_particle_ids)
filament_vels_z = vels_z[match_mask]
print("Recovered valocities for particles in filaments")

match_mask = np.isin(ids_z, wall_particle_ids)
wall_vels_z = vels_z[match_mask]
print("Recovered valocities for particles in walls")

match_mask = np.isin(ids_z, void_particle_ids)
void_vels_z = vels_z[match_mask]
print("Recovered valocities for particles in voids")

In [ ]:
node_speeds_z = np.sqrt(node_vels_z[:,0]**2 + node_vels_z[:,1]**2 + node_vels_z[:,2]**2)
filament_speeds_z = np.sqrt(filament_vels_z[:,0]**2 + filament_vels_z[:,1]**2 + filament_vels_z[:,2]**2)
wall_speeds_z = np.sqrt(wall_vels_z[:,0]**2 + wall_vels_z[:,1]**2 + wall_vels_z[:,2]**2)
void_speeds_z = np.sqrt(void_vels_z[:,0]**2 + void_vels_z[:,1]**2 + void_vels_z[:,2]**2)

In [ ]:
plt.hist(node_speeds_z, bins="scott", color = "orange", density=True, histtype = "step",label = "Nodes")
plt.hist(filament_speeds_z, bins="scott", color = "g", density=True, histtype = "step",label = "Filaments")
plt.hist(wall_speeds_z, bins="scott", color = "r", density=True, histtype = "step",label = "Walls")
plt.hist(void_speeds_z, bins="scott", color = "b", density=True, histtype = "step",label = "Voids")
plt.xlabel(r"km $\sqrt{a}$ / s")
plt.ylabel("PDF")
plt.grid()
plt.title("Speed distribution at z=20.05")
plt.legend();

In [ ]:
iidd = filament_particle_ids[66]

match_mask = np.isin(ids_z, iidd)
velocity_z = vels_z[match_mask]
print("Recovered valocities one particle in a filament at z=20.05")
print(velocity_z)

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    velss = part['Velocities']
    match_mask = np.isin(idss, iidd)
    velocity = velss[match_mask][0]
    vs.append(velocity)

vs = np.array(vs)

In [ ]:
snaps = np.arange(100)
diff = vs - velocity_z
diff_mag = np.sqrt(diff[:,0]**2 + diff[:,1]**2 + diff[:,2]**2)
plt.plot(snaps, diff_mag,c="b")
plt.grid();


In [ ]:
acc = np.zeros((99,3))

for i in range(99):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Fake Acceleration")

ax = fig.add_subplot(3,1,1)
ax.plot(snaps, acc[:,0],c="b")
ax.set_title("X")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(snaps, acc[:,1],c="r")
ax.set_title("Y")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(snaps, acc[:,2],c="g")
ax.grid()
ax.set_title("Z");

In [ ]:
fig = plt.figure(figsize=(10,8))
fig.suptitle("Deviation from initial velocity")

ax = fig.add_subplot(3,1,1)
ax.plot(snaps, diff[:,0],c="b")
ax.set_title("X")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(snaps, diff[:,1],c="r")
ax.set_title("Y")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(snaps, diff[:,2],c="g")
ax.grid()
ax.set_title("Z");

In [ ]:
iidd = filament_particle_ids[1234]

match_mask = np.isin(ids_z, iidd)
velocity_z = vels_z[match_mask]
print("Recovered valocities one particle in a filament at z=20.05")
print(velocity_z)

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    match_mask = np.isin(idss, iidd)
    velocity = part['Velocities'][match_mask][0]
    vs.append(velocity)

vs = np.array(vs)

In [ ]:
acc = np.zeros((100,3))

for i in range(99):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Fake Acceleration")

ax = fig.add_subplot(3,1,1)
ax.plot(snaps, acc[:,0],c="b")
ax.set_title("X")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(snaps, acc[:,1],c="r")
ax.set_title("Y")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(snaps, acc[:,2],c="g")
ax.grid()
ax.set_title("Z");

In [ ]:
iidd = filament_particle_ids

match_mask = np.isin(ids_z, iidd)
velocities_z = vels_z[match_mask]
mean_v_z = np.mean(velocities_z, axis = 0)
print("Recovered avg velocity of particles that are now in a filament at z=20.05")
print(mean_v_z)

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    velss = part['Velocities']
    match_mask = np.isin(idss, iidd)
    velocities = velss[match_mask]
    mean_v = np.mean(velocities, axis = 0)
    vs.append(mean_v)

vs = np.array(vs)
print(np.shape(vs))

In [ ]:
acc = np.zeros((100,3))

for i in range(99):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Mean fake acceleration")

ax = fig.add_subplot(3,1,1)
ax.plot(snaps, acc[:,0],c="b")
ax.set_title("X")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(snaps, acc[:,1],c="r")
ax.set_title("Y")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(snaps, acc[:,2],c="g")
ax.grid()
ax.set_title("Z");

In [ ]:
red = [ 20.05, 14.99, 11.98, 10.98, 10.00, 9.39, 9.00, 8.45, 8.01, 7.60, 7.24, 7.01, 6.49, 6.01,
    5.85, 5.53, 5.23, 5.00, 4.66, 4.43, 4.18, 4.01, 3.71, 3.49,
    3.28, 3.01, 2.90, 2.73, 2.58, 2.44, 2.32, 2.21, 2.10, 2.00,
    1.90, 1.82, 1.74, 1.67, 1.60, 1.53, 1.50, 1.41, 1.36, 1.30,
    1.25, 1.21, 1.15, 1.11, 1.07, 1.04, 1.00, 0.95, 0.92, 0.89,
    0.85, 0.82, 0.79, 0.76, 0.73, 0.70, 0.68, 0.64, 0.62, 0.60,
    0.58, 0.55, 0.52, 0.50, 0.48, 0.46, 0.44, 0.42, 0.40, 0.38,
    0.36, 0.35, 0.33, 0.31, 0.30, 0.27, 0.26, 0.24, 0.23, 0.21,
    0.20, 0.18, 0.17, 0.15, 0.14, 0.13, 0.11, 0.10, 0.08, 0.07,
    0.06, 0.05, 0.03, 0.02, 0.01, 0.00
]

In [ ]:
acc = np.zeros((100,3))

for i in range(99):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Mean fake acceleration for filament particles")

ax = fig.add_subplot(3,1,1)
ax.plot(red, acc[:,0],c="b")
ax.set_title("X")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(red, acc[:,1],c="r")
ax.set_title("Y")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"Speed in km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(red, acc[:,2],c="g")
ax.grid()
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"Speed in km $\sqrt{a} / s^2$")
ax.set_title("Z")

fig.tight_layout();

In [ ]:
iidd = node_particle_ids

match_mask = np.isin(ids_z, iidd)
velocities_z = vels_z[match_mask]
mean_v_z = np.mean(velocities_z, axis = 0)
print("Recovered avg velocity of particles that are now in a filament at z=20.05")
print(mean_v_z)

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    velss = part['Velocities']
    match_mask = np.isin(idss, iidd)
    velocities = velss[match_mask]
    mean_v = np.mean(velocities, axis = 0)
    vs.append(mean_v)

vs = np.array(vs)
print(np.shape(vs))

In [ ]:
acc = np.zeros((100,3))

for i in range(99):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Mean fake acceleration for node particles")

ax = fig.add_subplot(3,1,1)
ax.plot(red, acc[:,0],c="b")
ax.set_title("X")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(red, acc[:,1],c="r")
ax.set_title("Y")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"Speed in km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(red, acc[:,2],c="g")
ax.grid()
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"Speed in km $\sqrt{a} / s^2$")
ax.set_title("Z")

fig.tight_layout();

In [ ]:
iidd = wall_particle_ids

match_mask = np.isin(ids_z, iidd)
velocities_z = vels_z[match_mask]
mean_v_z = np.mean(velocities_z, axis = 0)
print("Recovered avg velocity of particles that are now in a filament at z=20.05")
print(mean_v_z)

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    velss = part['Velocities']
    match_mask = np.isin(idss, iidd)
    velocities = velss[match_mask]
    mean_v = np.mean(velocities, axis = 0)
    vs.append(mean_v)

vs = np.array(vs)
print(np.shape(vs))

In [ ]:
acc = np.zeros((100,3))

for i in range(99):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Mean fake acceleration for wall particles")

ax = fig.add_subplot(3,1,1)
ax.plot(red, acc[:,0],c="b")
ax.set_title("X")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(red, acc[:,1],c="r")
ax.set_title("Y")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(red, acc[:,2],c="g")
ax.grid()
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.set_title("Z")

fig.tight_layout();

In [ ]:
iidd = void_particle_ids

match_mask = np.isin(ids_z, iidd)
velocities_z = vels_z[match_mask]
mean_v_z = np.mean(velocities_z, axis = 0)
print("Recovered avg velocity of particles that are now in a filament at z=20.05")
print(mean_v_z)

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    velss = part['Velocities']
    match_mask = np.isin(idss, iidd)
    velocities = velss[match_mask]
    mean_v = np.mean(velocities, axis = 0)
    vs.append(mean_v)

vs = np.array(vs)
print(np.shape(vs))

In [ ]:
acc = np.zeros((99,3))

for i in range(100):
    acc[i] = vs[i+1] - vs[i]

fig = plt.figure(figsize=(10,8))
fig.suptitle("Mean fake acceleration for void particles")

ax = fig.add_subplot(3,1,1)
ax.plot(red, acc[:,0],c="b")
ax.set_title("X")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,2)
ax.plot(red, acc[:,1],c="r")
ax.set_title("Y")
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.grid()

ax = fig.add_subplot(3,1,3)
ax.plot(red, acc[:,2],c="g")
ax.grid()
ax.invert_xaxis()
ax.set_xlabel("Redshift z")
ax.set_ylabel(r"km $\sqrt{a} / s^2$")
ax.set_title("Z")

fig.tight_layout();

In [ ]:
vs = []

for i in range(100):
    s_id = i
    part = il.snapshot.loadSubset(basePath, s_id, 'dm', ['ParticleIDs', 'Velocities'])
    idss = part['ParticleIDs']
    velss = part['Velocities']
    abss = (velss[:,0]**2 + velss[:,1]**2 + velss[:,2]**2)**0.5
    mean_v = np.mean(abss)
    vs.append(mean_v)

vs = np.array(vs)
print(np.shape(vs))

In [ ]:
snaps = np.arange(100)
diff = vs - vs[0]
#diff_mag = np.sqrt(diff[:,0]**2 + diff[:,1]**2 + diff[:,2]**2)
plt.plot(snaps, diff,c="b")
plt.grid();
